# பாடம் 11 - மாடல் சூழல் நெறிமுறை (MCP)

**மாடல் சூழல் நெறிமுறை (MCP)** என்பது ஓர் திறந்த தரநிலை ஆகும், இது ஏஜென்ட்களுக்கு இயக்கநேரத்தில் கருவிகள், வளங்கள் மற்றும் தரவுத்தளங்களை தானாகக் கண்டறிந்து பயன்படுத்த உதவுகிறது. ஒரு ஏஜென்டில் கருவிகளை கடுமையாக நிரல்படுத்துவதற்குப் பதிலாக, MCP ஏஜென்ட்களுக்கு தேவைக்கேற்ப திறன்களை வெளிப்படுத்தும் வெளியே உள்ள சேவையகங்களுடன் இணைக்க அனுமதிக்கிறது.

இந்த பாடத்தில், நீங்கள் கற்றுக்கொள்ளப்போகிறீர்கள்:
- MCP என்ன மற்றும் அது ஏஜென்ட் அமைப்புகளுக்கு ஏன் முக்கியம்
- MCP கிளையன்ட்-சர்வர் கட்டமைப்பு எவ்வாறு செயல்படுகிறது
- MCP-பாணி கருவி கண்டறிதலை பயன்படுத்தும் ஏஜென்ட்களை எப்படி உருவாக்குவது


## அமைப்பு

**முன் நிபந்தனைகள்:**
- Azure AI Foundry திட்டம் செயல்படுத்தப்பட்ட மாதிரியுடன்
- அங்கீகாரணத்திற்காக `az login` ஐ இயக்கவும்


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## மாடல் சூழல் நெறிமுறை (MCP) என்பது என்ன?

MCP என்பது AI முகவர்கள் வெளியுறு கருவிகள் மற்றும் தரவுத் ஆதாரங்களுடன் தொடர்பு கொண்டு அவற்றை கண்டுபிடிக்கவும் தொடர்பு கொள்ளவும் பயன்படும் ஒரு நிலையான முறையை வரையறுக்கிறது:

- **MCP சேவையகம்**: ஒரு நிலையான நெறிமுறை மூலம் கருவிகள், வளங்கள் மற்றும் ப்ராம்ப்ட்களை வெளிப்படுத்துகிறது
- **MCP கிளையன்ட்**: சர்வர்களுடன் இணைந்து கிடைக்கும் திறன்களை ஓட்டுநேரத்தில் கண்டுபிடிக்கும் முகவர்
- **டையனமிக் கண்டுபிடிப்பு**: முகவர்களுக்கு முன்கூட்டியே நிரலிடப்பட்ட கருவிகள் தேவையில்லை — அவர்கள் ஓட்டுநேரத்தில் கிடைக்கும்வற்றை கண்டுபிடிக்கிறார்கள்

இது புதிய திறன்களை முகவர் குறியீட்டை மாற்றாமலே சேர்க்கக்கூடிய விரிவாக்கக்கூடிய முகவர் அமைப்புகளை உருவாக்குவதற்கு சக்திவாய்ந்ததாகும்.


## MCP எப்படி செயல்படுகிறது

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ஏஜென்ட் (MCP client) ஒரு MCP server க்கு இணைகிறது
2. சர்வர் கிடைக்கும் கருவிகளின் பட்டியல் மற்றும் அவற்றின் வடிவமைப்புகளுடன் பதிலளிக்கிறது
3. பிறகு ஏஜென்ட் தன் விவேசனையின் போது கண்டுபிடிக்கப்பட்ட எந்த கருவியையும் அழைக்கலாம்
4. முடிவுகள் அதே நெறிமுறை வழியாக திரும்ப செல்கின்றன


## MCP கருவி கண்டுபிடிப்பை சிமுலேட் செய்வது

நிஜ MCP சர்வர் ஒரு இயங்கும் சர்வர் செயலியை தேவைப்படுத்துவதால், MCP-இன் இணைக்கப்பட்ட வசதி சேவை வழங்கும் அம்சங்களை நகலாக்கும் வகையில் `@tool` செயல்பாடுகளைப் பயன்படுத்தி இந்த மாதிரியை காண்பிக்கிறோம்.

உற்பத்தியில், இக்கருவிகள் உள்ளூரில் வரையறுக்கப்படுவதற்குப் பதிலாக MCP சர்வரிலிருந்து தானாகவே கண்டுபிடிக்கப்படும்.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ஸ்டைல் கருவிகளுடன் ஒரு ஏஜென்டை உருவாக்குதல்


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## உற்பத்தி சூழலில் MCP

உற்பத்தி சூழலில், MCP வலிமையான நடைமுறைகளை சாத்தியமாக்குகிறது:

- **Dynamic tool discovery**: ஏஜென்டுகள் MCP சேவையகங்களுக்கு இணைந்து இயக்க நேரத்தில் கருவிகளை கண்டறிகின்றன
- **Decoupled architecture**: கருவி வழங்குநர்கள் ஏஜென்டிலிருந்து சுதந்திரமாக புதுப்பிக்க முடியும்
- **Cross-organization sharing**: அணிகள் MCP சேவையகங்கள் மூலம் திறன்களை வெளிப்படுத்தலாம்; அவற்றை எந்த ஏஜென்டும் பயன்படுத்தலாம்
- **Microsoft Agent Framework support**: MAF-இல் `mcp` ஒருங்கிணைப்பின் மூலம் உள்ளமைக்கப்பட்ட MCP கிளையன்ட் ஆதரவு உள்ளது

MAF உடன் ஒரு உண்மையான MCP சேவையகத்தை பயன்படுத்த, நீங்கள் `hosted_mcp_tool()` அல்லது MCP கிளையன்ட் ஒருங்கிணைப்பின் மூலம் இணைக்கலாம்.

**மேலும் அறிந்துகொள்ள:**
- [MCP விவரக்குறிப்பு](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ஆதரவு](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## சுருக்கம்

இந்தப் பாடத்தில் நீங்கள் கற்றுக்கொண்டவை:
- **MCP** என்பது ஏஜென்டுகள் மற்றும் கருவி வழங்குநர்களுக்கிடையில் இயக்கநேரத்தில் கருவிகளை கண்டறிதலுக்கான திறந்த தரநிலை
- **கிளையன்ட்-சர்வர் கட்டமைப்பு** ஏஜென்டுகள் இயக்கநேரத்தில் திறன்களை கண்டறிய அனுமதிக்கிறது
- **MCP** குறியீடு மாற்றங்களின்றி கருவிகளைச் சேர்க்க அனுமதிக்கும் விரிவாக்கக்கூடிய, பிரித்தெடுக்கப்பட்ட ஏஜென்ட் அமைப்புகளை சாத்தியப்படுத்தும்
- Microsoft Agent Framework உற்பத்தி பயன்பாட்டிற்காக **உள்ளமைக்கப்பட்ட MCP ஆதரவை** வழங்குகிறது


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
மறுப்பு அறிவிப்பு:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயன்றாலும், தானியங்கி மொழிபெயர்ப்புகளில் தவறுகள் அல்லது துல்லியத்தின்மைகள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனிக்கவும். அசல் ஆவணம் அதன் மூல மொழியிலேயே அதிகாரபூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவலுக்கு தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கின்றோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டினால் ஏற்படும் எந்தவொரு தவறான புரிதல்களுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பேற்கமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
